In [5]:
# 06_analysis_and_figures.ipynb -- dose-response, per-perturbation metric-judgment divergence, manuscript figures/tables

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Dataframe I/O that needs no parquet engine: use Parquet if one is installed, else CSV.
def _parquet_ok():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod); return True
        except Exception:
            pass
    return False
PARQUET = _parquet_ok()
def write_df(df: pd.DataFrame, stem: Path):
    # 'stem' is a path without extension; returns the path actually written.
    if PARQUET:
        p = stem.with_suffix(".parquet"); df.to_parquet(p); return p
    p = stem.with_suffix(".csv"); df.to_csv(p, index=False); return p
def df_exists(stem: Path) -> bool:
    return stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists()
def read_df(stem: Path) -> pd.DataFrame:
    pq, cv = stem.with_suffix(".parquet"), stem.with_suffix(".csv")
    if pq.exists(): return pd.read_parquet(pq)
    if cv.exists(): return pd.read_csv(cv)
    raise FileNotFoundError(f"{stem}.parquet/.csv")

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 06: turn judge scores into the paper's figures and tables ---
src = PROC / "judge_scores"
if not df_exists(src):
    print("[skip] run 05_judge_evaluation first (no judge_scores).")
else:
    s = read_df(src)
    for c in ("qualitative_adequacy", "ragas_faithfulness"):
        s[c] = pd.to_numeric(s[c], errors="coerce")

    # (a) Dose-response on P1: judge qualitative-adequacy vs technical faithfulness.
    p1 = s[s["ptype"].isin(["P0_original", "P1_inconsistency"])]
    dr = (p1.groupby("level")[["qualitative_adequacy", "ragas_faithfulness"]].mean().reset_index())
    save_table(dr.round(3), "06_dose_response")
    fig, ax = plt.subplots(figsize=(3.4, 2.6))
    ax.plot(dr["level"], dr["qualitative_adequacy"], marker="o", color="0.2",
            label="LLM judge (qual.)")
    ax.plot(dr["level"], dr["ragas_faithfulness"] * 5, marker="s", color="0.6",
            linestyle="--", label="Faithfulness (x5)")
    ax.set_xlabel("P1 inconsistency level (notches)"); ax.set_ylabel("Score")
    ax.legend(frameon=False); sns.despine(ax=ax)
    save_fig(fig, "06_dose_response")

    # (b) Per-perturbation contrast: the headline figure. For each perturbation, how much does the
    #     judge's qualitative score drop vs how much the technical faithfulness metric drops?
    order = ["P0_original", "P1_inconsistency", "P2_remove_qualitative",
             "P3_factual_error", "P4_direction_flip", "P5_verbosity"]
    present = [p for p in order if p in set(s["ptype"])]
    g = (s[s["ptype"].isin(present)].groupby("ptype")
         [["qualitative_adequacy", "ragas_faithfulness"]].mean().reindex(present).reset_index())
    g["ragas_faithfulness_x5"] = g["ragas_faithfulness"] * 5
    save_table(g.round(3), "06_perturbation_contrast")
    labels = [p.split("_", 1)[0] for p in g["ptype"]]
    x = np.arange(len(labels)); w = 0.38
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.bar(x - w / 2, g["qualitative_adequacy"], w, color="0.25", label="LLM judge (qual.)")
    ax.bar(x + w / 2, g["ragas_faithfulness_x5"], w, color="0.7",
           edgecolor="0.2", label="Faithfulness (x5)")
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Score (1-5)")
    ax.set_ylim(0, 5.6)
    ax.legend(frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0))
    sns.despine(ax=ax)
    save_fig(fig, "06_perturbation_contrast")

    # (c) Legacy P2-only contrast table (kept for continuity).
    p2 = s[s["ptype"].isin(["P0_original", "P2_remove_qualitative"])]
    save_table(p2.groupby("ptype")[["qualitative_adequacy", "ragas_faithfulness"]]
               .mean().round(3).reset_index(), "06_p2_contrast")

    # (d) Significance tests. Each perturbation is paired to its P0 counterpart on
    #     (case_id, model_tag). For P1 we use its strongest level (3) for a clean 1:1 pairing.
    #     Three Wilcoxon signed-rank tests per perturbation:
    #       - p_judge   : does the judge's qualitative score drop vs P0?
    #       - p_metric  : does the faithfulness metric drop vs P0?
    #       - p_diverge : does the JUDGE drop MORE than the metric? (judge_drop vs metric_drop,
    #                     both on the 1-5 scale; metric scaled x5). Positive divergence = judge
    #                     more sensitive to the defect than the technical metric.
    #     The divergence family is Holm-corrected for multiple comparisons.
    from scipy import stats

    base = (s[s["ptype"] == "P0_original"][["case_id", "model_tag",
            "qualitative_adequacy", "ragas_faithfulness"]]
            .rename(columns={"qualitative_adequacy": "q0", "ragas_faithfulness": "f0"}))

    def rep_rows(pt):
        sub = s[s["ptype"] == pt]
        if pt == "P1_inconsistency" and "level" in sub.columns and (sub["level"] == 3).any():
            sub = sub[sub["level"] == 3]
        return sub

    def paired_dz(x, y):
        d = (np.asarray(x, float) - np.asarray(y, float))
        d = d[~np.isnan(d)]
        sd = d.std(ddof=1) if len(d) > 1 else 0.0
        return float(d.mean() / sd) if sd > 0 else np.nan

    def wilcox_p(a, b):
        a = np.asarray(a, float); b = np.asarray(b, float)
        m = ~(np.isnan(a) | np.isnan(b)); a, b = a[m], b[m]
        if len(a) < 5 or np.allclose(a, b):
            return np.nan, len(a)
        try:
            return float(stats.wilcoxon(a, b).pvalue), len(a)
        except Exception:
            return np.nan, len(a)

    rows = []
    for pt in ["P1_inconsistency", "P2_remove_qualitative", "P3_factual_error",
               "P4_direction_flip", "P5_verbosity"]:
        if pt not in set(s["ptype"]):
            continue
        sub = rep_rows(pt).merge(base, on=["case_id", "model_tag"], how="inner")
        if sub.empty:
            continue
        jq, j0 = sub["qualitative_adequacy"], sub["q0"]
        fq, f0 = sub["ragas_faithfulness"], sub["f0"]
        judge_drop = j0 - jq                 # 1-5 scale
        metric_drop = (f0 - fq) * 5          # faithfulness on the same 1-5 scale
        p_j, n = wilcox_p(jq, j0)
        p_m, _ = wilcox_p(fq, f0)
        p_d, _ = wilcox_p(judge_drop, metric_drop)
        rows.append(dict(
            ptype=pt, n=n,
            judge_P0=round(float(j0.mean()), 3), judge_var=round(float(jq.mean()), 3),
            judge_drop=round(float(judge_drop.mean()), 3), judge_dz=round(paired_dz(jq, j0), 3),
            p_judge=p_j,
            metricx5_P0=round(float(f0.mean() * 5), 3), metricx5_var=round(float(fq.mean() * 5), 3),
            metric_drop=round(float(metric_drop.mean()), 3), p_metric=p_m,
            divergence=round(float((judge_drop - metric_drop).mean()), 3), p_divergence=p_d))
    sig = pd.DataFrame(rows)

    if not sig.empty:
        # Holm-Bonferroni across the divergence tests.
        pv = sig["p_divergence"].to_numpy(float)
        order_idx = np.argsort(np.where(np.isnan(pv), np.inf, pv))
        m_tests = int((~np.isnan(pv)).sum())
        holm = np.full(len(pv), np.nan)
        running = 0.0
        for rank, idx in enumerate(order_idx):
            if np.isnan(pv[idx]):
                continue
            running = max(running, min((m_tests - rank) * pv[idx], 1.0))
            holm[idx] = round(running, 4)
        sig["p_divergence_holm"] = holm
        for c in ("p_judge", "p_metric", "p_divergence"):
            sig[c] = sig[c].round(4)

        def star(p):
            if p is None or (isinstance(p, float) and np.isnan(p)):
                return "ns"
            return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        sig["sig_divergence"] = sig["p_divergence_holm"].map(star)
    save_table(sig, "06_significance")
    print("wrote 06 figures/tables | significance rows:", len(sig))

wrote 06 figures/tables | significance rows: 5
